# S1-PR-07 — Real REES46 Flower Smoke

**NON-SCIENTIFIC INTEGRATION SMOKE.** This notebook summarizes the real T1 → client → Flower/FedAvg path. It is not the official GRU, final R2a, R1 comparison, QR, or TEST evaluation.


## 1. Scope and privacy boundary

- Frozen C1 + T1 TRAIN/VALIDATION TaskExamples only.
- Reuses S1-PR-06 opaque client identity and deterministic sampler.
- No raw events.
- No `sealed_test`.
- Displays public-safe hashes, counts, digests and aggregate metrics only.


In [ ]:
from pathlib import Path
import json

CONFIG = Path("config/fl_real_smoke.v1.json")
INPUTS = Path("docs/evidence/s1-pr-07/task_example_inputs.v1.json")
SUMMARY = Path("docs/evidence/s1-pr-07/fl_real_smoke_summary.v1.json")

config = json.loads(CONFIG.read_text(encoding="utf-8"))
inputs = json.loads(INPUTS.read_text(encoding="utf-8"))
summary = json.loads(SUMMARY.read_text(encoding="utf-8"))

print("Purpose:", summary["purpose"])
print("Status:", summary["status"])
print("Scientific claims:", summary.get("scientific_claims", []))


## 2. Frozen inputs

Read from public-safe evidence. This notebook does not open private parquets.


In [ ]:
{
    "cohort_sha256": inputs["cohort_manifest"]["sha256"],
    "t1_train_rows": inputs["t1_train"]["rows"],
    "t1_train_sha256": inputs["t1_train"]["sha256"],
    "t1_validation_rows": inputs["t1_validation"]["rows"],
    "t1_validation_sha256": inputs["t1_validation"]["sha256"],
    "category_count": inputs["vocabulary"]["category_count"],
    "base_c1_clients": inputs["base_client_manifest"]["client_count"],
    "t1_smoke_eligible_clients": inputs["t1_smoke_eligible_client_count"],
    "raw_events_used": inputs["raw_events_used"],
    "sealed_test_accessed": inputs["sealed_test_accessed"],
}


## 3. Data flow

```text
Frozen T1 TaskExamples
→ S1-PR-06 client identity
→ task-eligible client counts
→ deterministic sampler
→ tiny real per-client T1 slices
→ Phase1Batch (zero-history smoke)
→ LocalTrainerCore + FlowerLocalAdapter
→ Flower/Ray FedAvg
→ server validation smoke + structured result
```


## 4. Federated rounds

Only selection digests and aggregate counts are shown.


In [ ]:
[
    {
        "round": r["server_round"],
        "selection_digest": r["selection_digest"],
        "selected": r["selected_client_count"],
        "successful": r["successful_client_count"],
        "contributing_examples": r["contributing_examples"],
        "mean_local_train_loss": r["mean_local_train_loss"],
        "max_abs_diff": r["max_abs_diff"],
        "oracle_pass": r["aggregation_oracle_pass"],
    }
    for r in summary["rounds"]
]


## 5. Server validation smoke history

Diagnostics only — not research headline metrics.


In [ ]:
summary["validation_history"]


## 6. Reproducibility


In [ ]:
summary["reproducibility"]


## 7. Final checks


In [ ]:
{
    "status": summary["status"],
    "global_model_changed": summary["global_model_changed"],
    "aggregation_oracle_pass": summary["aggregation_oracle_pass"],
    "reproducibility_pass": summary["reproducibility"]["pass"],
    "raw_events_used": summary["raw_events_used"],
    "sealed_test_accessed": summary["sealed_test_accessed"],
    "scientific_claims": summary.get("scientific_claims", []),
}


## 8. Limitations and next step

This smoke deliberately uses a zero-history representation and the existing trainer-contract stub model. It proves the real federated pipeline, not final recommendation quality. The next scientific work uses the official model/evaluator. TEST remains sealed.
